In [1]:
!nvidia-smi

Sat Mar  7 13:20:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:07:00.0 Off |                    0 |
| N/A   36C    P0             58W /  400W |       4MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import os
import subprocess
from pathlib import Path

def batch_extract_vimedpet():
    # Define source and target directories
    source_dir = Path('/data24/xl693/ViMedPET')
    target_dir = Path('/data24/xl693/unzipViMedPET')
    
    # Create target directory if it doesn't exist
    target_dir.mkdir(parents=True, exist_ok=True)
    
    # Recursively find all main .zip files in all subfolders
    # We ignore the .z01, .z02 files in the search because 7z will pull them automatically
    zip_files = sorted(source_dir.rglob('*.zip'))
    
    if not zip_files:
        print("No .zip files found in the specified directory.")
        return

    print(f"Found {len(zip_files)} main zip archives. Starting batch extraction to {target_dir}...\n")
    
    for zip_file in zip_files:
        print(f"Extracting {zip_file.name}...")
        
        # Construct the 7-Zip command
        # 'x' extracts files with full paths
        # '-o' specifies the output directory (note: no space between -o and the path)
        # '-y' automatically answers 'yes' to any overwrite prompts
        cmd = ['7z', 'x', str(zip_file), f'-o{target_dir}', '-y']
        
        try:
            # Run the extraction command
            result = subprocess.run(cmd, capture_output=True, text=True, check=True)
            print(f"  -> Successfully extracted {zip_file.name}")
        except subprocess.CalledProcessError as e:
            print(f"  -> Error extracting {zip_file.name}!")
            print(f"     Details: {e.stderr}")
        except FileNotFoundError:
            print("  -> ERROR: '7z' command not found. Please ensure p7zip is installed on your server.")
            break

if __name__ == '__main__':
    batch_extract_vimedpet()

Found 6 main zip archives. Starting batch extraction to /data24/xl693/unzipViMedPET...

Extracting PETCT_2017.zip...
  -> Successfully extracted PETCT_2017.zip
Extracting PETCT_2018.zip...
  -> Successfully extracted PETCT_2018.zip
Extracting PETCT_2019.zip...
  -> Successfully extracted PETCT_2019.zip
Extracting PETCT_2023_1.zip...
  -> Successfully extracted PETCT_2023_1.zip
Extracting PETCT_2023_2.zip...
  -> Successfully extracted PETCT_2023_2.zip
Extracting PETCT_2023_3.zip...
  -> Successfully extracted PETCT_2023_3.zip


In [4]:
import os
from pathlib import Path
import numpy as np
import scipy.ndimage as ndi
import SimpleITK as sitk

# ============================================================
# 1. Core SimpleITK Physical Space Utilities
# ============================================================

def zyx_to_xyz_spacing(spacing_zyx):
    """Convert numpy spacing (Z, Y, X) to SimpleITK convention (X, Y, Z)."""
    return (float(spacing_zyx[2]), float(spacing_zyx[1]), float(spacing_zyx[0]))

def create_sitk_image(array_zyx, spacing_zyx):
    """Creates a SimpleITK image with explicit physical spacing."""
    img = sitk.GetImageFromArray(array_zyx.astype(np.float32))
    img.SetSpacing(zyx_to_xyz_spacing(spacing_zyx))
    img.SetOrigin((0.0, 0.0, 0.0))
    img.SetDirection((1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0))
    return img

def estimate_pet_spacing_from_shape(pet_shape, ct_shape, ct_spacing_zyx):
    """
    Estimates PET spacing assuming CT and PET cover the same physical FOV width.
    Formula: Spacing_PET = Spacing_CT * (Shape_CT / Shape_PET)
    """
    pet_shape_np = np.array(pet_shape, dtype=np.float64)
    ct_shape_np = np.array(ct_shape, dtype=np.float64)
    ct_spacing_np = np.array(ct_spacing_zyx, dtype=np.float64)
    
    pet_spacing = ct_spacing_np * (ct_shape_np / pet_shape_np)
    return tuple(float(v) for v in pet_spacing)

# ============================================================
# 2. Advanced Morphology Masking & Normalization
# ============================================================

def get_ct_body_mask(ct_array, ct_thresh=300):
    """Builds a robust 3D body mask using morphology to close holes and remove artifacts."""
    mask = ct_array > ct_thresh
    mask = ndi.binary_closing(mask, structure=np.ones((3, 5, 5), dtype=bool))
    mask = ndi.binary_fill_holes(mask)
    
    labels, num = ndi.label(mask)
    if num == 0:
        return np.zeros_like(mask, dtype=bool)
        
    counts = np.bincount(labels.ravel())
    counts[0] = 0
    keep = np.argmax(counts)
    
    mask = labels == keep
    mask = ndi.binary_opening(mask, structure=np.ones((3, 3, 3), dtype=bool))
    return mask.astype(bool)

def get_bounding_box_from_mask(mask, pad=(3, 8, 8)):
    """Computes a compact 3D bounding box from the morphology mask."""
    zz, yy, xx = np.where(mask)
    if len(zz) == 0:
        return None
        
    zmin, zmax = max(0, zz.min() - pad[0]), min(mask.shape[0], zz.max() + pad[0] + 1)
    ymin, ymax = max(0, yy.min() - pad[1]), min(mask.shape[1], yy.max() + pad[1] + 1)
    xmin, xmax = max(0, xx.min() - pad[2]), min(mask.shape[2], xx.max() + pad[2] + 1)
    
    return slice(zmin, zmax), slice(ymin, ymax), slice(xmin, xmax)

def robust_zscore(volume, mask=None, eps=1e-8):
    """Calculates Z-score normalization strictly within the body mask to prevent background skew."""
    vol = volume.astype(np.float32)
    vals = vol[mask > 0] if (mask is not None and np.any(mask)) else vol.reshape(-1)
    
    mu = np.mean(vals)
    sigma = max(float(np.std(vals)), eps)
    return (vol - mu) / sigma

# ============================================================
# 3. Physical Downsampling & Registration
# ============================================================

def resample_to_shared_grid(sitk_image, target_size, target_spacing_xyz, is_ct=False):
    """Resamples the image into a mathematically predefined physical grid."""
    resampler = sitk.ResampleImageFilter()
    resampler.SetSize(target_size)
    resampler.SetOutputSpacing(target_spacing_xyz)
    resampler.SetOutputOrigin((0.0, 0.0, 0.0))
    resampler.SetOutputDirection((1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0))
    resampler.SetInterpolator(sitk.sitkLinear)
    resampler.SetDefaultPixelValue(-1000.0 if is_ct else 0.0)
    
    return resampler.Execute(sitk_image)

def register_arrays_mi(ct_array: np.ndarray, pet_array: np.ndarray, spacing_zyx: tuple) -> np.ndarray:
    """
    Registers a PET array to a CT array using Mattes Mutual Information.
    Uses an Affine transform $A \in \mathbb{R}^{4 \times 4}$ constrained by physical spacing.
    """
    fixed_image = create_sitk_image(ct_array, spacing_zyx)
    moving_image = create_sitk_image(pet_array, spacing_zyx)
    
    registration_method = sitk.ImageRegistrationMethod()
    registration_method.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    registration_method.SetMetricSamplingStrategy(registration_method.RANDOM)
    registration_method.SetMetricSamplingPercentage(0.1)
    registration_method.SetInterpolator(sitk.sitkLinear)
    
    registration_method.SetOptimizerAsGradientDescent(
        learningRate=1.0, 
        numberOfIterations=200, 
        convergenceMinimumValue=1e-6, 
        convergenceWindowSize=10
    )
    registration_method.SetOptimizerScalesFromPhysicalShift()
    
    initial_transform = sitk.CenteredTransformInitializer(
        fixed_image, 
        moving_image, 
        sitk.AffineTransform(3), 
        sitk.CenteredTransformInitializerFilter.GEOMETRY
    )
    registration_method.SetInitialTransform(initial_transform, inPlace=False)
    
    try:
        final_transform = registration_method.Execute(fixed_image, moving_image)
        print(f"        -> Optimizer stopped: {registration_method.GetOptimizerStopConditionDescription()}")
    except Exception as e:
        print(f"        -> Registration failed: {e}. Returning original PET.")
        return pet_array
    
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(fixed_image)
    resampler.SetInterpolator(sitk.sitkLinear)
    # Pad with the minimum value of the normalized PET array to avoid artificial bright edges
    resampler.SetDefaultPixelValue(float(np.min(pet_array))) 
    resampler.SetTransform(final_transform)
    
    registered_pet_image = resampler.Execute(moving_image)
    return sitk.GetArrayFromImage(registered_pet_image)

# ============================================================
# 4. Strict Ordered Preprocessing Pipeline
# ============================================================

def preprocess_and_save(pet_path, ct_path, save_path, orig_ct_spacing_zyx=(3.0, 1.0, 1.0), target_spacing_zyx=(3.0, 2.0, 2.0)):
    # ---------------- Step 0: Load Arrays and Flip ----------------
    pet_array = np.load(pet_path).astype(np.float32)[::-1, :, :]
    ct_array = np.load(ct_path).astype(np.float32)[::-1, :, :]
    print(f"  -> RAW Arrays - PET: {pet_array.shape}, CT: {ct_array.shape}")
    
    # ---------------- Step 1: Assign Initial Physical Space ----------------
    orig_pet_spacing_zyx = estimate_pet_spacing_from_shape(pet_array.shape, ct_array.shape, orig_ct_spacing_zyx)
    print(f"      -> [Step 1] Initialized Physical Spaces. Est. PET Spacing (ZYX): {orig_pet_spacing_zyx}")
    
    sitk_ct = create_sitk_image(ct_array, orig_ct_spacing_zyx)
    sitk_pet = create_sitk_image(pet_array, orig_pet_spacing_zyx)
    
    # ---------------- Step 2: Joint Physical Target Resampling ----------------
    print(f"      -> [Step 2] Downsampling BOTH to a shared physical grid ZYX={target_spacing_zyx}...")
    target_xyz = zyx_to_xyz_spacing(target_spacing_zyx)
    
    orig_size_ct = sitk_ct.GetSize()
    orig_spacing_ct = sitk_ct.GetSpacing()
    target_size = [
        int(round(orig_size_ct[0] * (orig_spacing_ct[0] / target_xyz[0]))),
        int(round(orig_size_ct[1] * (orig_spacing_ct[1] / target_xyz[1]))),
        int(round(orig_size_ct[2] * (orig_spacing_ct[2] / target_xyz[2])))
    ]
    
    sitk_ct_downsampled = resample_to_shared_grid(sitk_ct, target_size, target_xyz, is_ct=True)
    sitk_pet_downsampled = resample_to_shared_grid(sitk_pet, target_size, target_xyz, is_ct=False)
    
    ct_final_grid = sitk.GetArrayFromImage(sitk_ct_downsampled).astype(np.float32)
    pet_final_grid = sitk.GetArrayFromImage(sitk_pet_downsampled).astype(np.float32)
    
    # ---------------- Step 3: Body Masking & Cropping ----------------
    print("      -> [Step 3] Morphology-based body cropping...")
    body_mask = get_ct_body_mask(ct_final_grid, ct_thresh=300)
    bbox = get_bounding_box_from_mask(body_mask, pad=(3, 3, 3))
    
    if bbox is not None:
        ct_crop, pet_crop, body_mask_crop = ct_final_grid[bbox], pet_final_grid[bbox], body_mask[bbox]
    else:
        ct_crop, pet_crop, body_mask_crop = ct_final_grid, pet_final_grid, body_mask
        
    # ---------------- Step 4: Robust Masked Normalization ----------------
    print("      -> [Step 4] Independent Z-Score Normalization...")
    ct_norm = robust_zscore(ct_crop, mask=body_mask_crop)
    pet_norm = robust_zscore(pet_crop, mask=body_mask_crop)
    
    # ---------------- Step 4.5: Affine Registration ----------------
    print("      -> [Step 4.5] Registering Normalized PET to CT...")
    pet_norm_registered = register_arrays_mi(ct_norm, pet_norm, spacing_zyx=target_spacing_zyx)
    
    # ---------------- Step 5: Stack and Save ----------------
    stacked = np.stack([pet_norm_registered, ct_norm], axis=0)
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(save_path, stacked)
    print(f"  -> Successfully Saved! Final Tensor Shape: {stacked.shape}\n" + "-"*60)

# ============================================================
# 5. Execution Loop
# ============================================================

def process_vimedpet_dataset():
    source_base = Path('/data24/xl693/unzipViMedPET')
    target_base = Path('/gluon4/xl693/PETCTfoundation/ViMedPET')
    
    orig_ct_spacing_zyx = (3.0, 1.0, 1.0)
    target_spacing_zyx = (3.0, 2.0, 2.0)
    
    print("Starting Mathematically Strict ViMedPET Preprocessing Pipeline (With Registration)...")
    print("=" * 70)
    
    for pet_dir in list(source_base.rglob('PET')):
        ct_dir = pet_dir.parent / 'CT'
        if not ct_dir.exists(): continue
            
        for pet_file in pet_dir.glob('*.npy'):
            ct_file = ct_dir / pet_file.name
            if not ct_file.exists(): continue
                
            clean_rel_path = Path(*[p for p in pet_file.relative_to(source_base).parts if p != 'PET'])
            print(f"Processing: {clean_rel_path.name}")
            try:
                preprocess_and_save(
                    pet_path=pet_file,
                    ct_path=ct_file,
                    save_path=target_base / clean_rel_path,
                    orig_ct_spacing_zyx=orig_ct_spacing_zyx,
                    target_spacing_zyx=target_spacing_zyx
                )
            except Exception as e:
                print(f"  [Error] Failed on {pet_file.name}: {e}\n" + "-"*60)

if __name__ == '__main__':
    process_vimedpet_dataset()

<>:95: SyntaxWarning: invalid escape sequence '\i'
<>:95: SyntaxWarning: invalid escape sequence '\i'
/tmp/ipykernel_2521049/3585712309.py:95: SyntaxWarning: invalid escape sequence '\i'
  """


Starting Mathematically Strict ViMedPET Preprocessing Pipeline (With Registration)...
Processing: day_20_patient_1124.npy
  -> RAW Arrays - PET: (299, 256, 256), CT: (299, 512, 512)
      -> [Step 1] Initialized Physical Spaces. Est. PET Spacing (ZYX): (3.0, 2.0, 2.0)
      -> [Step 2] Downsampling BOTH to a shared physical grid ZYX=(3.0, 2.0, 2.0)...
      -> [Step 3] Morphology-based body cropping...
      -> [Step 4] Independent Z-Score Normalization...
      -> [Step 4.5] Registering Normalized PET to CT...
        -> Optimizer stopped: GradientDescentOptimizerv4Template: Convergence checker passed at iteration 74.
  -> Successfully Saved! Final Tensor Shape: (2, 299, 209, 243)
------------------------------------------------------------
Processing: day_27_patient_1173.npy
  -> RAW Arrays - PET: (299, 256, 256), CT: (299, 512, 512)
      -> [Step 1] Initialized Physical Spaces. Est. PET Spacing (ZYX): (3.0, 2.0, 2.0)
      -> [Step 2] Downsampling BOTH to a shared physical grid ZYX